# Model evaluation

Inspect one run exported by `scripts/evaluate.py`. Select its directory below,
then run all cells. Charts use saved metrics; prediction is never rerun.

Use validation runs while choosing configurations. An untrained model only
checks the pipeline; reserve the test set for final evaluation.


In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RUN = ROOT / "results" / "RUN1"  # --- Select here an evaluation run ---

path = RUN / "metrics.json"
if not path.is_file():
    raise FileNotFoundError(f"No metrics at {path}. Run scripts/evaluate.py first.")
metrics = json.loads(path.read_text())
labels = metrics["confusion_matrix"]["labels"]
print(f"Run: {RUN.name} | Positions: {metrics['samples']:,}")
print(f"Accuracy: {metrics['accuracy']:.2%} | Macro F1: {metrics['macro_f1']:.3f}")
print(f"Analyzer elapsed time: {metrics['prediction_seconds']:.2f} s (includes loading)")
for name in ("model", "dataset"):
    print(f"{name.capitalize()}: {metrics['artifacts'][name]['path']}")


## Scores by class

Precision: how often predictions of a class are correct. Recall: how many actual
positions of that class were found. F1 combines both. Undefined scores are zero;
macro F1 includes all three classes, even those with no examples.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
width = 0.24
for offset, key in enumerate(("precision", "recall", "f1")):
    positions = [i + (offset - 1) * width for i in range(len(labels))]
    bars = ax.bar(positions, [metrics["per_class"][label][key] for label in labels],
                  width, label=key.capitalize())
    ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=8)
ax.set_xticks(range(len(labels)),
              [f"{label}\nn={metrics['per_class'][label]['support']:,}" for label in labels])
ax.set_ylim(0, 1.15)
ax.set_ylabel("Score")
ax.set_title(f"Per-class performance — {RUN.name}")
ax.legend(loc="upper right", ncols=3)
ax.spines[["top", "right"]].set_visible(False)
plt.show()


## Confusion matrix

Rows are actual classes; columns are predictions. The diagonal shows correct
predictions. Row percentages make errors comparable across class sizes.
A row with no examples is shown as zeros.


In [ ]:
counts = metrics["confusion_matrix"]["counts"]
percentages = [[100 * value / sum(row) if sum(row) else 0 for value in row]
               for row in counts]
fig, ax = plt.subplots(figsize=(6, 5), layout="constrained")
heatmap = ax.imshow(percentages, cmap="Blues", vmin=0, vmax=100)
for row in range(len(labels)):
    for column in range(len(labels)):
        percent = percentages[row][column]
        ax.text(column, row, f"{counts[row][column]:,}\n{percent:.1f}%",
                ha="center", va="center", color="white" if percent > 50 else "black")
ax.set_xticks(range(len(labels)), labels)
ax.set_yticks(range(len(labels)), labels)
ax.set_xlabel("Predicted class")
ax.set_ylabel("Actual class")
ax.set_title(f"Confusion matrix — {RUN.name}")
fig.colorbar(heatmap, ax=ax, label="% of actual class")
plt.show()


## Interpretation

Record the weakest class, the main confusion, and the next experiment here.
Compare configurations on the same validation dataset before selecting a model.


## Compare validation runs

Select exported validation runs in `RUNS` below. The validation file's SHA-256,
analyzer build, class order, and class supports must match. Test runs are
rejected against the declared validation file. Existing `RUN1` used test data:
keep it for inspection above, not for configuration selection.

Rows are ranked by macro F1, then worst-class recall. Timing is descriptive.
These are individual runs, not averages across training seeds. See the
[fixed protocol](../docs/evaluation.md#comparison-protocol-v1) before selecting a configuration.


In [ ]:
import sys
from html import escape
from IPython.display import HTML, display

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.comparison import load_comparison

RUNS = []  # Select validation run directory names.
VALIDATION_DATASET = ROOT / "dataset" / "splits" / "validation.txt"

comparison = (load_comparison([ROOT / "results" / name for name in RUNS],
                              VALIDATION_DATASET) if RUNS else [])
if not comparison:
    print("Select validation runs in RUNS, then rerun this section.")
else:
    rows = []
    for report in comparison:
        m = report["metrics"]
        values = [escape(report["run"]), f"{m['samples']:,}", f"{m['accuracy']:.2%}",
                  f"{m['macro_f1']:.3f}", f"{report['worst_recall']:.3f}",
                  f"{m['prediction_seconds']:.2f}"]
        rows.append("<tr>" + "".join(f"<td>{value}</td>" for value in values) + "</tr>")
    headings = ("Run", "Positions", "Accuracy", "Macro F1", "Worst recall", "Elapsed (s)")
    display(HTML("<table><thead><tr>" + "".join(f"<th>{h}</th>" for h in headings)
                 + "</tr></thead><tbody>" + "".join(rows) + "</tbody></table>"))
    print("Validation SHA-256:", comparison[0]["metrics"]["artifacts"]["dataset"]["sha256"])
    print("Single-run timing includes startup and loading; it is not a speed ranking.")


In [ ]:
if comparison:
    fig, axes = plt.subplots(1, 2, figsize=(12, max(4, 0.45 * len(comparison) + 2)),
                             layout="constrained")
    names = [report["run"] for report in comparison]
    width = 0.24
    for ax, key in zip(axes, ("f1", "recall")):
        for offset, label in enumerate(labels):
            positions = [i + (offset - 1) * width for i in range(len(comparison))]
            values = [report["metrics"]["per_class"][label][key] for report in comparison]
            bars = ax.barh(positions, values, width, label=label)
            ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=8)
        ax.set_yticks(range(len(names)), names)
        ax.invert_yaxis()
        ax.set_xlim(0, 1.15)
        ax.set_xlabel("Score")
        ax.set_title("F1 by class" if key == "f1" else "Recall by class")
        ax.spines[["top", "right"]].set_visible(False)
    axes[0].legend(loc="upper center", bbox_to_anchor=(0.5, 1.18), ncols=3)
    plt.show()
